Question 2.1 Two normalisations of the coordinates are compared. Both translate the origin to landmark 27, which 
removes the position of the face in the image. The first divides by the inter-ocular distance d, the 
distance between landmarks 36 and 45 in the same frame:

x′k= (xk−x27)/d ,y′k= (yk−y27)/d ,    

d =√(( x45−x36)^2+( y45− y36)^2)

The second divides by the dimensions of the image, 1280 × 720 pixels:

x′k= (xk−x27)/1280 , y′k= (yk− y27)/720

Compare the two through the variability between actors in the neutral expression: for each 
normalisation, compute the median width of the mouth (the distance between landmarks 48 and 54) 
in the neutral videos of each actor, and the coefficient of variation of these 20 medians. Which 
normalisation leaves less variation between actors, and why? What does dividing x by 1280 and y by 
720 do to the geometry of the face? All the remaining tasks use the normalisation by d.

In [13]:
import numpy as np
import pandas as pd

df = pd.read_parquet("t1.1.gzip")
testIndex = pd.read_parquet("testIndex.gzip")

df = df[~df["video"].isin(testIndex["video"])]

# video names are like "01-01-01-01-01-01-01.csv"
neutral = df[df["video"].astype(str).str.split("-").str[2].eq("01")].copy()

neutral["actor"] = neutral["video"].str.replace(".csv", "", regex=False).str[-2:]

#d =√(( x45−x36)^2+( y45− y36)^2)
d = np.sqrt((neutral["x_45"] - neutral["x_36"])**2 + (neutral["y_45"] - neutral["y_36"])**2)

#x′k= (xk−x27)/d ,y′k= (yk−y27)/d  ("compute the median width of the mouth (the distance between landmarks 48 and 54)")
x48_d = (neutral["x_48"] - neutral["x_27"]) / d
y48_d = (neutral["y_48"] - neutral["y_27"]) / d
x54_d = (neutral["x_54"] - neutral["x_27"]) / d
y54_d = (neutral["y_54"] - neutral["y_27"]) / d

#x′k= (xk−x27)/1280 , y′k= (yk− y27)/720
x48_img = (neutral["x_48"] - neutral["x_27"]) / 1280
y48_img = (neutral["y_48"] - neutral["y_27"]) / 720
x54_img = (neutral["x_54"] - neutral["x_27"]) / 1280
y54_img = (neutral["y_54"] - neutral["y_27"]) / 720

#Mouth width = distance between landmarks 48 and 54, with each normalisation
neutral["width_d"]   = np.sqrt((x54_d - x48_d)**2 + (y54_d - y48_d)**2)
neutral["width_img"] = np.sqrt((x54_img - x48_img)**2 + (y54_img - y48_img)**2)

#median per actor
medians = neutral.groupby("actor")[["width_d", "width_img"]].median()
print(medians)

#Coefficient of variation = standard deviation / mean, of those 20 medians
cv_d   = medians["width_d"].std()   / medians["width_d"].mean()
cv_img = medians["width_img"].std() / medians["width_img"].mean()
print("CV with d:        ", cv_d)
print("CV with image size:", cv_img)


        width_d  width_img
actor                     
02     0.566075   0.090233
03     0.624451   0.097635
04     0.611065   0.095907
05     0.619302   0.108878
06     0.615028   0.096501
07     0.637391   0.104304
08     0.613624   0.094383
09     0.569194   0.095751
10     0.591432   0.095975
11     0.627349   0.111028
12     0.551322   0.088065
13     0.621648   0.098018
14     0.615739   0.100030
15     0.637833   0.108186
18     0.608866   0.107025
19     0.628179   0.106243
20     0.596602   0.089660
21     0.559526   0.081565
22     0.578067   0.101290
23     0.642346   0.103130
CV with d:         0.04556682175548994
CV with image size: 0.0785016760874476


Normalising by the inter-ocular distance d leaves less variation between actors: the coefficient of variation of the 20 median mouth widths is 0.046, against 0.079 when dividing by the image size.

The reason is scale. The size of a face in the image depends on how far the actor sits from the camera, and not on the expression. The inter-ocular distance d is measured on the same face, so it grows and shrinks with the face. Dividing the mouth width by d cancels this effect and leaves the width relative to the face. Dividing by 1280 and 720 divides every actor by the same constant, so the differences in scale between actors are kept.

Dividing x by 1280 and y by 720 scales the two axes by different factors, so the face is deformed. The normalisation by d scales both axes by the same factor, so the shape of the face is preserved. It is used in all the remaining tasks.